##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemma 2 和 LlamaCpp 入門

[Gemma](https://ai.google.dev/gemma) 是 Google 推出的一系列輕量級、最先進的開源語言模式。 Gemma 模型採用與創建 Gemini 模型相同的研究和技術構建而成，是文本到文本、僅限解碼器的大語言模型 (LLM)，提供英語版本，具有開放權重、預訓練變體和指令調整變體。
Gemma 模型非常適合各種文本生成任務，包括問答、總結和推論。它們相對較小的尺寸使得可以將它們部署在資源有限的環境中，例如筆記型電腦、桌上型電腦或雲端基礎設施，從而實現對最先進人工智慧模型的民主化訪問，並幫助促進每個人的創新。
[llama.cpp](https://github.com/ggerganov/llama.cpp) 是 Meta AI 的 LLaMA 和其他大型語言模型架構的 C++ 實現，旨在在本地計算機或Google Colab 等環境中實現高效性能。它使您能夠執行大型語言模型，而無需大量計算資源。
為了使 llama.cpp 的使用更容易，
[llama-cpp-python](https://github.com/abetlen/llama-cpp-python) 為 C++ library 提供 Python 綁定。這使您能夠享受 `llama.cpp` 的效能最佳化，同時受益於 Python 的簡單性和靈活性。使用 llama-cpp-python，您可以獲得方便的 API 來載入模型、生成文字和自訂 inference 參數。
在此notebook中，您將學習如何在Google Colab環境中使用`llama.cpp`執行Gemma 2個模型。您將安裝必要的軟體包、設定模型並執行範例prompt。
<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/Gemma/[Gemma_2]Using_with_LlamaCpp.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

## 設定

### 選擇 Colab runtime
要完成本教學，您需要擁有 Colab runtime 以及足夠的資源來執行 Gemma 模型。在這種情況下，您可以使用 T4 GPU：
1. 在 Colab 視窗的右上角，選擇 **▾（其他連接選項）**。
2. 選擇**更改 runtime 類型**。
3. 在 **硬體加速器** 下，選擇 **T4 GPU**。

### Gemma設置

**在深入學習本教學之前，讓我們先設定Gemma：**
1. **Hugging Face 帳戶：** 如果您還沒有帳戶，您可以點選[此處](https://huggingface.co/join) 建立免費的Hugging Face 帳戶。
2. **Gemma 模型存取：** 前往 [Gemma 模型頁面](https://huggingface.co/collections/google/gemma-2-release-667d6600fd5220e7b967f315) 並接受使用條件。
3. **Colab 和 Gemma 功能：** 對於本教學，您需要一個 Colab runtime 並具有足夠的資源來處理 Gemma 2B 模型。開始Colab 會話時選擇適當的runtime。
4. **Hugging Face token：** 透過點選[此處](https://huggingface.co/settings/tokens) 產生Hugging Face 存取權限（最好是`write` 權限）token。在本教學的後面部分，您將需要這個token。

**完成這些步驟後，您就可以進入下一部分，在 Colab 環境中設定環境變數。 **

### 設定您的 HF token

將您的 Hugging Face token 新增至 Colab Secrets manager 以安全地儲存它。
1. 開啟 Google Colab notebook 並點選左側面板中的 🔑 Secrets 標籤。 <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
2. 建立一個新的secret，名稱為`HF_TOKEN`。
3. 將token 金鑰複製/貼上到`HF_TOKEN` 的值輸入框中。
4. 切換左側的按鈕以允許notebook 存取secret。


In [ ]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

### 安裝依賴項

您需要安裝一些 Python 套件和依賴項才能與 HuggingFace 以及 `llama-cpp-python` 進行互動並執行模型。 [此處](https://abetlen.github.io/llama-cpp-python/whl/cu122/llama-cpp-python/) 尋找一些支援 CUDA 12.2 的版本。
執行以下cell來安裝或升級它：

In [ ]:
# The huggingface_hub library allows us to download models and other files from Hugging Face.
!pip install --upgrade -q huggingface_hub

# The llama-cpp-python library allows us to leverage GPUs
!pip install llama-cpp-python==0.2.90 \
  -q -U --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

### 登入Hugging Face Hub

接下來，您必須使用您的存取權限token 登入Hugging Face Hub。這將使我們能夠下載 Gemma 模型。

In [ ]:
from huggingface_hub import login

login(os.environ["HF_TOKEN"])

### 下載Gemma 2模型
登入後，您可以從Hugging Face下載Gemma 2個模型檔案。 [Gemma 2 模型](https://huggingface.co/google/gemma-2-2b-GGUF) 提供 **GGUF** 格式，該格式針對與 `llama.cpp` 和 Llamafile 等相容工具的使用進行了最佳化。

In [ ]:
from huggingface_hub import hf_hub_download

# Specify the repository and filename
repo_id = 'google/gemma-2-2b-GGUF'  # Repository containing the GGUF model
filename = '2b_pt_v2.gguf'  # The GGUF model file

# Download the model file to the current directory
hf_hub_download(repo_id=repo_id, filename=filename, local_dir='.')

### 使用`llama.cpp`執行Gemma 2模型

可以使用 `llama.cpp` 二進位檔案直接執行 Gemma 2 模型，而無需依賴 Python 綁定。這種方法允許您直接在 C++ 層級工作，並且對於整合到非Python 應用程式或效能最佳化非常有用。
首先，您需要下載支援Gemma 2的`llama.cpp` [二進位](https://github.com/ggerganov/llama.cpp/releases/tag/b3496)。
注意：下載的二進位檔案不支援 GPU。

In [ ]:
# Download the prebuilt llama.cpp binary (with Gemma 2 support)
!wget -O llama.cpp.zip https://github.com/ggerganov/llama.cpp/releases/download/b3496/llama-b3496-bin-ubuntu-x64.zip
!mkdir llama.cpp && unzip -qq llama.cpp.zip -d llama.cpp

現在，您可以使用`llama.cpp` 二進位檔案來執行帶有範例prompt 的Gemma 2 模型。

In [ ]:
# Define your prompt
prompt = "Q: I have 3 apples and I eat 2. How many apples do I have left?\nA:"
model_path = "2b_pt_v2.gguf"

# Run the model using llama.cpp
!./llama.cpp/build/bin/llama-cli -m "{model_path}" -p "{prompt}" -n 3 \
  --color --chat-template gemma --temp 0.7

產生回應可能需要一些時間，但很快您就會了解如何使用 `llama-cpp-python` Python library 來利用 Colab 的內建 GPU。

### 使用`llama-cpp-python`執行Gemma 2模型

您將透過從 HuggingFace 載入預先訓練的 Gemma 2 模型，使用 `llama-cpp-python` library 初始化 Llama 模型。以下是程式碼各部分的作用：
- `model_path`：模型的路徑。
- `verbose`：在模型載入期間停用詳細日誌記錄以獲得更清晰的輸出。
- `n_gpu_layers`：設定 GPU 加速。值 `-1` 表示它將使用盡可能多的 GPU 層。
- `chat_format`：設定預期的輸入和輸出格式以符合Gemma模型的聊天格式。

這透過從 HuggingFace 載入模型權重來設定環境以將 Gemma 2 語言模型與 `llama.cpp` 一起使用。它準備模型以產生與 Gemma 規範相容的格式的文字。

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    verbose=False,
    n_gpu_layers=-1,
    chat_format="gemma",
)

### 與Gemma模型交互
最後，您 prompt Gemma 2 模型並產生回應。
若要執行此操作並在生成時串流傳輸輸出，請在呼叫 `llm` 函數時設定 `stream=True` 並使用 `for` 循環迭代響應；在 `print` 函數中使用 `end=''` 可確保文字流不會在每個區塊後設定新功能streaming。

In [ ]:
prompt = "What are large language models?"

# Increase max_tokens if you need longer responses
output = llm(prompt, max_tokens=256, temperature=0.7, stream=True)

# Iterate over the streaming response and print them
for response in output:
    print(response['choices'][0]['text'], end='', flush=True)

恭喜！您已在Colab 環境中使用`llama.cpp` 成功設定了Gemma 2 模型。現在您可以試驗該模型、生成文字並探索其功能。